# Episode 7: DPO: teaching a GPT from preferences

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep007-dpo.ipynb)

With direct preference optimisation, our fine-tuned model learns from pairs of its own stories, where the one that uses more of the words asked for is preferred, and we measure what it gains and what it costs.

[The video](https://youtu.be/aQ4WxQfB2S4) · [The episode's page](https://mvgpt.nenuadrian.com/ep007-dpo/) · [The docs](https://nenuadrian.github.io/mvGPT/dpo/) · [The code at `ep007/end`](https://github.com/nenuadrian/mvGPT/tree/ep007/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep007/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


fetch("sft")

## The data

In [ ]:
!python data.py  # TinyStories, the 10,000-word vocabulary, and the stories as token ids

## Pairs of stories
Two stories for each of 2,000 instructions (and 200 held out), the one that uses more of the
requested words preferred.

In [ ]:
!python preferences.py 2000 200

## DPO
Ten minutes of learning from the pairs.

In [ ]:
!python dpo.py 10
# or, to skip the ten minutes, the video's: fetch("dpo")

## Measuring it, and stories before and after

In [ ]:
!python evaluate.py checkpoints/sft/model.pt checkpoints/dpo/model.pt --out results/dpo
!python sample.py checkpoints/sft/model.pt "Words: rain, cake, jump. Story:" -n 1 --seed 3
!python sample.py checkpoints/dpo/model.pt "Words: rain, cake, jump. Story:" -n 1 --seed 3

Next: [episode 8, PPO: reinforcement learning for a GPT](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep008-ppo.ipynb)